<a href="https://colab.research.google.com/github/Dunk0725/GIS-26/blob/NJ-Youth-Football-GIS/PS3_NJ_Youth_Football_GIS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PS3 - Geographic and Organizational Structure of New Jersey Youth Football

**Project:** New Jersey Youth and Amateur Sports Association (NJYASA) planning research  
**Assignment:** PS3 - First Draft of the Final Project  
**Course:** Geographic Information Systems for Public Sector  
**Author:** David  
**Year:** 2026

## 1. Project purpose

This project examines how community youth football is geographically and organizationally structured across New Jersey and how those existing patterns can inform a proposed statewide NJYASA framework.

The first three maps follow one sequence:

1. **Where are documented youth football organizations located?**
2. **How are those organizations currently connected through existing leagues?**
3. **How do those same organizations fall within the proposed NJYASA North, Central, and South regions?**

The analysis is intended to support planning. It does **not** treat mapped organizations as NJYASA members, claim that current leagues are ineffective, or assume that the proposed regional structure is already optimal.


## 2. Response to PS2 feedback

The PS2 feedback emphasized slowing down, using fewer maps, interpreting results more deeply, considering measures beyond simple counts, checking AI-assisted code carefully, and documenting AI assistance.

This PS3 draft responds by:

- limiting the main story to a small number of connected maps;
- using the same analysis-ready organization records throughout the core maps;
- adding county-by-league, region, and league-by-region descriptive analyses;
- preserving data-quality and eligibility limitations instead of treating every mapped point as fully field-verified;
- separating existing league affiliations from the proposed NJYASA regional structure;
- reserving public funding/resource comparisons for later maps only if they directly advance the NJYASA research question; and
- maintaining an AI-assistance appendix with prompts and outputs used in developing the notebook.

**Important:** verbal and written PS2 comments should be kept here and answered individually before final submission.


## 3. Research question and unit of analysis

### Research question

**How is community youth football geographically and organizationally structured across New Jersey, and how could those existing patterns inform the proposed regional structure of NJYASA?**

### Unit of analysis

The unit of analysis is a **documented community youth football organization**.

Each record is represented geographically by its recorded home-field or facility location. A point does not represent an individual team, an athlete residence, an organization's complete service area, or every facility used by that organization.

### Current quality-control rule

The audited preparation workflow contains 265 football organizations after excluding the documented cheer-only record P035. One organization, P181, remains excluded from the analysis-ready dataset pending organization verification. The present analysis-ready total is therefore **264 organizations** unless that research decision is formally updated in the preparation workflow.


## 4. Data sources and reproducibility

This notebook is designed to load its analytical inputs from public online sources.

### Core sources

- **NJ county boundaries:** New Jersey Office of GIS ArcGIS Feature Service.
- **Youth football locations:** public PS0 research CSV on GitHub during development.
- **PS3 master organization attributes:** a planned public CSV containing league, county, facility, coordinate, and QA fields for the analysis-ready organizations.

### Development note

The existing public PS0 CSV contains coordinates and QA fields but does not include league affiliation. Until the PS3 master CSV is published, Map 1 and the proposed region analysis can run from the PS0 fallback; the league-affiliation cells will stop with a clear message.

Before submission, the fallback should be removed and the PS3 notebook should load one approved public analysis file directly from GitHub.


In [ ]:
# -------------------------------------------------------
# PS3 - SECTION 4.1
# PYTHON ENVIRONMENT
# -------------------------------------------------------

import io
import requests
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

from IPython.display import display

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)

print("pandas:", pd.__version__)
print("geopandas:", gpd.__version__)


In [ ]:
# -------------------------------------------------------
# PS3 - SECTION 4.2
# PUBLIC DATA SOURCE URLS
# -------------------------------------------------------

COUNTY_URL = (
    "https://services2.arcgis.com/XVOqAjTOJ5P6ngMu/"
    "arcgis/rest/services/NJ_Counties_3424/FeatureServer/0/query?"
    "where=1%3D1&outFields=*&returnGeometry=true&outSR=4326&f=geojson"
)

# Current public PS0 fallback - 265 records / 264 eligible for mapping
PS0_FOOTBALL_URL = (
    "https://raw.githubusercontent.com/"
    "Dunk0725/GIS-26/"
    "NJ-Youth-Football-GIS/"
    "PS0_NJ_Youth_Football_Locations_2026.csv"
)

# Planned PS3 master file. Publish this file to the same GitHub branch.
PS3_MASTER_URL = (
    "https://raw.githubusercontent.com/"
    "Dunk0725/GIS-26/"
    "NJ-Youth-Football-GIS/"
    "PS3_NJ_Youth_Football_Analysis_Ready_2026.csv"
)

print("County source:", COUNTY_URL)
print("PS3 master source:", PS3_MASTER_URL)


In [ ]:
# -------------------------------------------------------
# PS3 - SECTION 4.3
# LOAD NJ COUNTY BOUNDARIES
# -------------------------------------------------------

counties = gpd.read_file(COUNTY_URL)

required_county_fields = {"COUNTY", "geometry"}
missing_county_fields = required_county_fields - set(counties.columns)

if missing_county_fields:
    raise ValueError(
        f"County layer is missing required fields: {sorted(missing_county_fields)}"
    )

counties = counties[["COUNTY", "geometry"]].copy()
counties = counties.to_crs("EPSG:4326")
counties["COUNTY"] = counties["COUNTY"].astype("string").str.strip()

assert len(counties) == 21, f"Expected 21 NJ counties, found {len(counties)}"
assert counties["COUNTY"].is_unique
assert counties.geometry.notna().all()

print("NJ counties loaded:", len(counties))
print("CRS:", counties.crs)
display(counties.head())


In [ ]:
# -------------------------------------------------------
# PS3 - SECTION 4.4
# LOAD YOUTH FOOTBALL RESEARCH DATA
# -------------------------------------------------------


def read_public_csv(url, timeout=60):
    response = requests.get(url, timeout=timeout)
    response.raise_for_status()
    return pd.read_csv(io.BytesIO(response.content), dtype={"Program_ID": "string"})


try:
    football = read_public_csv(PS3_MASTER_URL)
    DATA_MODE = "PS3 master dataset"
except Exception as exc:
    print("PS3 master file is not public yet.")
    print("Development fallback: loading the current PS0 public CSV.")
    print("Reason:", type(exc).__name__)
    football = read_public_csv(PS0_FOOTBALL_URL)
    DATA_MODE = "PS0 development fallback"

football["Program_ID"] = football["Program_ID"].astype("string").str.strip()

assert football["Program_ID"].notna().all()
assert football["Program_ID"].is_unique

print("Data mode:", DATA_MODE)
print("Rows loaded:", len(football))
print("Columns:", len(football.columns))
print()
print(football.columns.tolist())


In [ ]:
# -------------------------------------------------------
# PS3 - SECTION 4.5
# CREATE THE ANALYSIS-READY ORGANIZATION TABLE
# -------------------------------------------------------

analysis = football.copy()

# If the public master contains eligibility status, apply it.
if "Analysis_Eligibility" in analysis.columns:
    included_status = "Included - subject to documented QA limitations"
    eligible = analysis["Analysis_Eligibility"].eq(included_status)

    print("Eligibility counts:")
    print(analysis["Analysis_Eligibility"].value_counts(dropna=False))

    analysis = analysis.loc[eligible].copy()

# Coordinates are required for the maps.
for column in ["Latitude", "Longitude"]:
    if column not in analysis.columns:
        raise ValueError(f"Missing required coordinate field: {column}")
    analysis[column] = pd.to_numeric(analysis[column], errors="coerce")

if analysis[["Latitude", "Longitude"]].isna().any().any():
    bad = analysis.loc[
        analysis[["Latitude", "Longitude"]].isna().any(axis=1),
        ["Program_ID", "Canonical_Program_Name", "Latitude", "Longitude"]
    ]
    display(bad)
    raise ValueError("One or more eligible organizations have missing coordinates.")

football_points = gpd.GeoDataFrame(
    analysis,
    geometry=gpd.points_from_xy(
        analysis["Longitude"],
        analysis["Latitude"]
    ),
    crs="EPSG:4326"
)

assert football_points["Program_ID"].is_unique
assert football_points.geometry.notna().all()
assert football_points.geometry.is_valid.all()

print("Organizations in current analysis:", len(football_points))
print("Unique Program_ID values:", football_points["Program_ID"].nunique())
print("CRS:", football_points.crs)

if DATA_MODE == "PS0 development fallback":
    assert len(football_points) == 264, (
        "The PS0 fallback previously contained 264 eligible records. "
        "If that has changed, review the updated public file before changing this assertion."
    )


### 4.6 Data-integrity check before mapping

The county recorded during research and the county derived from GIS are kept separate. The spatial county is calculated from the mapped point; it is not copied from the researched county field.

Missing values are not treated as successful matches. The purpose of this check is to identify records requiring review before county or regional statistics are calculated.


In [ ]:
# -------------------------------------------------------
# PS3 - SECTION 4.6
# SPATIALLY ASSIGN EACH ORGANIZATION TO AN NJ COUNTY
# -------------------------------------------------------

county_polygons = counties.rename(columns={"COUNTY": "GIS_County"}).copy()

# Remove any older join products before creating a fresh spatial join.
clean_points = football_points.drop(
    columns=["GIS_County", "Spatial_County", "index_right"],
    errors="ignore"
).copy()

football_by_county = gpd.sjoin(
    clean_points,
    county_polygons[["GIS_County", "geometry"]],
    how="left",
    predicate="within"
)

# A normal NJ point should match exactly one county.
if len(football_by_county) != len(clean_points):
    raise ValueError(
        "The spatial join changed the number of organization records. "
        "Check for duplicate or overlapping county polygons."
    )

assert football_by_county["Program_ID"].is_unique

missing_gis_county = football_by_county["GIS_County"].isna().sum()

print("Organizations after county join:", len(football_by_county))
print("Missing GIS county:", int(missing_gis_county))

if missing_gis_county:
    display(
        football_by_county.loc[
            football_by_county["GIS_County"].isna(),
            ["Program_ID", "Canonical_Program_Name", "Latitude", "Longitude"]
        ]
    )

# If a researched county is available, compare it to the independent GIS result.
if "Primary_County" in football_by_county.columns:
    researched = (
        football_by_county["Primary_County"]
        .astype("string")
        .str.strip()
        .str.casefold()
        .replace("", pd.NA)
    )

    spatial = (
        football_by_county["GIS_County"]
        .astype("string")
        .str.strip()
        .str.casefold()
        .replace("", pd.NA)
    )

    comparable = researched.notna() & spatial.notna()
    football_by_county["County_Match"] = pd.NA
    football_by_county.loc[comparable, "County_Match"] = (
        researched.loc[comparable].eq(spatial.loc[comparable])
    )

    print()
    print("County comparison:")
    print(football_by_county["County_Match"].value_counts(dropna=False))

    mismatches = football_by_county.loc[
        football_by_county["County_Match"].eq(False),
        [
            "Program_ID",
            "Canonical_Program_Name",
            "Primary_County",
            "GIS_County",
            "Latitude",
            "Longitude"
        ]
    ]

    if len(mismatches):
        print("County differences requiring review:")
        display(mismatches)


## 5. Map 1 - Documented youth football organizations by county

### Question

**Where are the documented community youth football organizations in New Jersey?**

This map establishes the geographic foundation before league affiliations or the proposed NJYASA regions are introduced.

The county count is based on the GIS-derived county containing each mapped point. The point remains the organization-level observation.


In [ ]:
# -------------------------------------------------------
# PS3 - MAP 1
# DOCUMENTED YOUTH FOOTBALL ORGANIZATIONS BY COUNTY
# -------------------------------------------------------

county_counts = (
    football_by_county.groupby("GIS_County")
    .size()
    .rename("Organization_Count")
    .reset_index()
)

map1_counties = (
    counties.rename(columns={"COUNTY": "GIS_County"})
    .merge(county_counts, on="GIS_County", how="left")
)

map1_counties["Organization_Count"] = (
    map1_counties["Organization_Count"].fillna(0).astype(int)
)

fig, ax = plt.subplots(figsize=(10, 12))

map1_counties.plot(
    ax=ax,
    color="#eef1f4",
    edgecolor="#4d5963",
    linewidth=0.8,
    zorder=1
)

football_by_county.plot(
    ax=ax,
    color="#173f5f",
    markersize=20,
    alpha=0.80,
    edgecolor="white",
    linewidth=0.35,
    zorder=3
)

# County labels and organization counts.
label_points = map1_counties.copy()
label_points["label_geometry"] = label_points.geometry.representative_point()

for _, row in label_points.iterrows():
    x = row["label_geometry"].x
    y = row["label_geometry"].y
    ax.text(
        x,
        y,
        f'{row["GIS_County"]}\n(n={row["Organization_Count"]})',
        ha="center",
        va="center",
        fontsize=7.3,
        color="#222222",
        zorder=4
    )

ax.set_title(
    "Documented Community Youth Football Organizations in New Jersey\n"
    "by GIS-Derived County",
    fontsize=15,
    fontweight="bold",
    pad=16
)

ax.set_axis_off()

fig.text(
    0.5,
    0.035,
    f"Organizations mapped: {len(football_by_county)} | "
    "Points represent documented organization home-field/facility locations.",
    ha="center",
    fontsize=8.5
)

plt.tight_layout(rect=[0, 0.055, 1, 1])
plt.show()

print("Organization counts by county:")
display(
    county_counts.sort_values(
        ["Organization_Count", "GIS_County"],
        ascending=[False, True]
    ).reset_index(drop=True)
)


### 5.1 Interpretation checklist for Map 1

After running the map, write a short interpretation based on the actual results:

- Which counties have the largest and smallest documented organization counts?
- Are organizations concentrated in particular parts of the state?
- Are there counties where organizations appear geographically dispersed rather than clustered?
- Does an equal number of counties imply an equal number of organizations? No; that question will be tested directly in the regional analysis.
- What does the map **not** measure? It does not show athlete participation, demand, facility quality, travel burden, or complete service areas.

Do not write the final interpretation until the table and map have been run and checked.


In [ ]:
# -------------------------------------------------------
# PS3 - DESCRIPTIVE STATISTICS SUPPORTING MAP 1
# HISTOGRAM: ORGANIZATIONS PER COUNTY
# -------------------------------------------------------

all_counties_for_hist = (
    counties[["COUNTY"]]
    .rename(columns={"COUNTY": "GIS_County"})
    .merge(county_counts, on="GIS_County", how="left")
)

all_counties_for_hist["Organization_Count"] = (
    all_counties_for_hist["Organization_Count"].fillna(0).astype(int)
)

counts = all_counties_for_hist["Organization_Count"]

fig, ax = plt.subplots(figsize=(8, 5))

bins = np.arange(counts.min(), counts.max() + 2) - 0.5
ax.hist(counts, bins=bins, edgecolor="black")

ax.set_title("Distribution of Documented Youth Football Organizations Across NJ Counties")
ax.set_xlabel("Organizations in county")
ax.set_ylabel("Number of counties")

plt.tight_layout()
plt.show()

print(counts.describe())


## 6. Existing league affiliations

### Question

**How are the documented organizations currently connected through existing leagues?**

League affiliation belongs to the organization, not to the county. A county can contain organizations from multiple leagues, and a league can extend across multiple counties.

For this reason, the organization points carry the league symbol/color while county polygons remain a geographic reference layer.


In [ ]:
# -------------------------------------------------------
# PS3 - SECTION 6.1
# VERIFY LEAGUE FIELDS
# -------------------------------------------------------

league_field = "League_Acronym"
league_name_field = "Current_League"

LEAGUE_READY = league_field in football_by_county.columns

if not LEAGUE_READY:
    print("League_Acronym is not present in the current public dataset.")
    print("Map 2 and the league matrices will be skipped for now.")
    print()
    print("Next step: publish PS3_NJ_Youth_Football_Analysis_Ready_2026.csv")
    print("from the audited analysis-ready dataset, then rerun the notebook.")
else:
    football_by_county[league_field] = (
        football_by_county[league_field]
        .astype("string")
        .str.strip()
        .replace("", pd.NA)
    )

    print("Organizations:", len(football_by_county))
    print("Organizations with league acronym:", football_by_county[league_field].notna().sum())
    print("Organizations without league acronym:", football_by_county[league_field].isna().sum())
    print("Distinct league acronyms:", football_by_county[league_field].nunique(dropna=True))

    print()
    print("League counts:")
    display(
        football_by_county[league_field]
        .value_counts(dropna=False)
        .rename("Organizations")
        .to_frame()
    )


In [ ]:
# -------------------------------------------------------
# PS3 - MAP 2
# EXISTING LEAGUE AFFILIATIONS
# -------------------------------------------------------

if not LEAGUE_READY:
    print("Map 2 skipped: the PS3 public master dataset has not been loaded yet.")
else:
    league_points = football_by_county.loc[
        football_by_county[league_field].notna()
    ].copy()

    league_values = sorted(league_points[league_field].unique())

    fig, ax = plt.subplots(figsize=(12, 12))

    counties.plot(
        ax=ax,
        color="#f4f4f4",
        edgecolor="#666666",
        linewidth=0.8,
        zorder=1
    )

    cmap = plt.get_cmap("tab20", max(len(league_values), 1))

    for idx, league in enumerate(league_values):
        subset = league_points.loc[
            league_points[league_field].eq(league)
        ]

        subset.plot(
            ax=ax,
            color=cmap(idx),
            markersize=28,
            alpha=0.90,
            edgecolor="white",
            linewidth=0.4,
            label=f"{league} (n={len(subset)})",
            zorder=3
        )

    county_labels = counties.copy()
    county_labels["label_geometry"] = county_labels.geometry.representative_point()

    for _, row in county_labels.iterrows():
        ax.text(
            row["label_geometry"].x,
            row["label_geometry"].y,
            row["COUNTY"],
            ha="center",
            va="center",
            fontsize=6.2,
            color="#444444",
            zorder=2
        )

    ax.set_title(
        "League Affiliations of Documented New Jersey Youth Football Organizations",
        fontsize=15,
        fontweight="bold",
        pad=16
    )

    ax.set_axis_off()

    ax.legend(
        title="Recorded league affiliation",
        bbox_to_anchor=(1.02, 1),
        loc="upper left",
        fontsize=8,
        title_fontsize=9,
        frameon=True
    )

    fig.text(
        0.5,
        0.03,
        "County boundaries are geographic reference areas; league affiliation is represented at the organization level.",
        ha="center",
        fontsize=8.5
    )

    plt.tight_layout(rect=[0, 0.05, 0.82, 1])
    plt.show()


In [ ]:
# -------------------------------------------------------
# PS3 - SECTION 6.2
# COUNTY x LEAGUE ORGANIZATION MATRIX
# -------------------------------------------------------

if not LEAGUE_READY:
    print("County-by-league matrix skipped until the PS3 public master dataset is available.")
else:
    county_league_matrix = pd.crosstab(
        football_by_county["GIS_County"],
        football_by_county[league_field],
        dropna=False
    )

    county_league_matrix["Total_Organizations"] = county_league_matrix.sum(axis=1)
    county_league_matrix["Distinct_Leagues"] = (
        county_league_matrix.drop(columns="Total_Organizations").gt(0).sum(axis=1)
    )

    county_league_matrix = county_league_matrix.sort_values(
        ["Distinct_Leagues", "Total_Organizations"],
        ascending=False
    )

    print("County-by-league organization matrix:")
    display(county_league_matrix)

    print("Counties with the greatest number of recorded leagues:")
    display(
        county_league_matrix[["Distinct_Leagues", "Total_Organizations"]]
        .head(10)
    )


### 6.3 Interpretation checklist for Map 2

Use the actual matrix and map to answer:

- Which leagues contain the largest numbers of documented organizations?
- Which counties contain organizations from multiple leagues?
- Which leagues extend across several counties?
- Are there geographic clusters associated with particular leagues?
- Which existing league relationships would a statewide NJYASA framework need to account for?

A league crossing a county or proposed regional boundary is not automatically a problem. It is a structural relationship that the proposed system would need to understand.


## 7. Proposed NJYASA regional structure

The proposed planning structure assigns each New Jersey county to one of three regions:

- **South:** Atlantic, Burlington, Camden, Cape May, Cumberland, Gloucester, Salem
- **Central:** Ocean, Hunterdon, Mercer, Middlesex, Monmouth, Somerset, Union
- **North:** Warren, Bergen, Essex, Hudson, Morris, Passaic, Sussex

Each region contains seven counties. The analysis below tests whether equal county counts also produce comparable numbers of documented football organizations or comparable existing league structures.


In [ ]:
# -------------------------------------------------------
# PS3 - SECTION 7.1
# DEFINE PROPOSED NJYASA REGIONS
# -------------------------------------------------------

region_counties = {
    "South": [
        "Atlantic", "Burlington", "Camden", "Cape May",
        "Cumberland", "Gloucester", "Salem"
    ],
    "Central": [
        "Ocean", "Hunterdon", "Mercer", "Middlesex",
        "Monmouth", "Somerset", "Union"
    ],
    "North": [
        "Warren", "Bergen", "Essex", "Hudson",
        "Morris", "Passaic", "Sussex"
    ]
}

county_to_region = {
    county: region
    for region, county_list in region_counties.items()
    for county in county_list
}

assert len(county_to_region) == 21
assert len(set(county_to_region)) == 21

expected_counties = set(counties["COUNTY"])
configured_counties = set(county_to_region)

if expected_counties != configured_counties:
    print("Missing from region configuration:", sorted(expected_counties - configured_counties))
    print("Unexpected in region configuration:", sorted(configured_counties - expected_counties))
    raise ValueError("The NJYASA region configuration does not cover the 21 NJ counties exactly once.")

football_by_county["NJYASA_Region"] = football_by_county["GIS_County"].map(county_to_region)

if football_by_county["NJYASA_Region"].isna().any():
    raise ValueError("One or more organizations could not be assigned to a proposed NJYASA region.")

print("Organizations by proposed NJYASA region:")
print(football_by_county["NJYASA_Region"].value_counts())


In [ ]:
# -------------------------------------------------------
# PS3 - MAP 3
# PROPOSED NJYASA NORTH / CENTRAL / SOUTH REGIONS
# -------------------------------------------------------

region_map = counties.copy()
region_map["NJYASA_Region"] = region_map["COUNTY"].map(county_to_region)

region_colors = {
    "South": "#c62828",
    "Central": "#f7f7f7",
    "North": "#1f4e8c"
}

fig, ax = plt.subplots(figsize=(10, 12))

for region in ["North", "Central", "South"]:
    region_map.loc[
        region_map["NJYASA_Region"].eq(region)
    ].plot(
        ax=ax,
        color=region_colors[region],
        edgecolor="#333333",
        linewidth=1.0,
        alpha=0.78,
        zorder=1
    )

football_by_county.plot(
    ax=ax,
    color="#111111",
    markersize=22,
    alpha=0.85,
    edgecolor="white",
    linewidth=0.35,
    zorder=3
)

# County labels.
region_labels = region_map.copy()
region_labels["label_geometry"] = region_labels.geometry.representative_point()

for _, row in region_labels.iterrows():
    ax.text(
        row["label_geometry"].x,
        row["label_geometry"].y,
        row["COUNTY"],
        ha="center",
        va="center",
        fontsize=6.2,
        color="#222222",
        zorder=2
    )

# Region labels and organization totals.
region_totals = (
    football_by_county["NJYASA_Region"]
    .value_counts()
    .rename("Organizations")
)

region_shapes = region_map.dissolve(by="NJYASA_Region")
region_shapes["label_geometry"] = region_shapes.geometry.representative_point()

for region, row in region_shapes.iterrows():
    p = row["label_geometry"]
    ax.text(
        p.x,
        p.y,
        f"{region.upper()}\n{int(region_totals.get(region, 0))} organizations",
        ha="center",
        va="center",
        fontsize=10,
        fontweight="bold",
        bbox=dict(facecolor="white", alpha=0.80, edgecolor="#333333", pad=3),
        zorder=5
    )

ax.set_title(
    "Proposed NJYASA North, Central, and South Regions\n"
    "with Documented Youth Football Organizations",
    fontsize=15,
    fontweight="bold",
    pad=16
)

ax.set_axis_off()

fig.text(
    0.5,
    0.035,
    "Regional boundaries are a proposed NJYASA planning structure; mapped organizations are not represented as NJYASA members.",
    ha="center",
    fontsize=8.5
)

plt.tight_layout(rect=[0, 0.055, 1, 1])
plt.show()


In [ ]:
# -------------------------------------------------------
# PS3 - SECTION 7.2
# REGIONAL ORGANIZATION TOTALS AND STATEWIDE SHARES
# -------------------------------------------------------

regional_summary = (
    football_by_county.groupby("NJYASA_Region")
    .agg(
        Organizations=("Program_ID", "nunique"),
        Counties=("GIS_County", "nunique")
    )
    .reset_index()
)

regional_summary["Statewide_Share_Pct"] = (
    regional_summary["Organizations"]
    / regional_summary["Organizations"].sum()
    * 100
).round(1)

regional_summary["Organizations_Per_County"] = (
    regional_summary["Organizations"] / regional_summary["Counties"]
).round(2)

regional_summary = regional_summary.set_index("NJYASA_Region").loc[
    ["North", "Central", "South"]
].reset_index()

print("Proposed NJYASA regional summary:")
display(regional_summary)


In [ ]:
# -------------------------------------------------------
# PS3 - SECTION 7.3
# LEAGUE x PROPOSED REGION MATRIX
# -------------------------------------------------------

if LEAGUE_READY:
    league_region_matrix = pd.crosstab(
        football_by_county[league_field],
        football_by_county["NJYASA_Region"]
    )

    # Keep region order consistent.
    for region in ["North", "Central", "South"]:
        if region not in league_region_matrix.columns:
            league_region_matrix[region] = 0

    league_region_matrix = league_region_matrix[["North", "Central", "South"]]
    league_region_matrix["Total"] = league_region_matrix.sum(axis=1)
    league_region_matrix["Regions_Represented"] = (
        league_region_matrix[["North", "Central", "South"]].gt(0).sum(axis=1)
    )

    league_region_matrix = league_region_matrix.sort_values(
        ["Regions_Represented", "Total"],
        ascending=False
    )

    print("Existing leagues across proposed NJYASA regions:")
    display(league_region_matrix)
else:
    print("League-region matrix will be available after the PS3 master dataset is published.")


### 7.4 Interpretation checklist for Map 3

Use the actual regional totals and league-region matrix to discuss:

- Does dividing New Jersey into seven counties per region also divide the documented organizations evenly?
- What percentage of the statewide organization inventory falls in each proposed region?
- Which existing leagues are concentrated in one proposed region?
- Which existing leagues cross two or three proposed regional boundaries?
- What administrative or coordination questions does that create for NJYASA?

Do not interpret uneven counts as proof that the proposed boundaries are wrong. Organization counts do not measure participation, travel distance, population, facility capacity, or administrative workload by themselves.


## 8. Maps 4 and 5 - reserved for the next NJYASA question

The final two maps should only be added if they advance the central NJYASA story.

Possible directions already supported by earlier research include:

- public recreation investment and documented football geography;
- youth-service or volunteer-support geography;
- youth football and high-school football facility relationships; or
- another operational feature of NJYASA that can be measured with defensible data.

The decision should be made from the research question, not from a desire to fill all five map slots.


## 9. Current limitations

- The organization inventory has not been established as an exhaustive census of every active New Jersey youth football organization.
- Home-field/facility points do not describe athlete residence, complete service area, or every field used by an organization.
- Most locations are geocoded address-level points; geocoding does not automatically establish field-level accuracy.
- Recorded league affiliations may differ in evidence quality or verified season.
- The NJYASA regional structure is proposed and has not been adopted by the organizations shown.
- Equal county counts do not imply equal numbers of organizations, participants, facilities, population, travel burden, or administrative workload.


## 10. AI assistance appendix - working record

**AI system:** ChatGPT / OpenAI

AI assistance is being used to help review prior notebook structure, debug Python/GeoPandas code, design reproducibility checks, and draft candidate code and explanatory text.

For the submitted assignment, preserve the actual prompts and outputs used. The final notebook should reflect code and interpretations that the author has reviewed, understood, and revised.

### Prompt record - PS3 notebook development

**Prompt:** "Let's start building out this Google Colab notebook please"

**Use of output:** Initial PS3 notebook structure, data-loading strategy, geographic QA, first three map workflows, descriptive tables, and interpretation checklists. All cells should be rerun in Google Colab and checked against the source data before submission.


## 11. Next development checkpoint

Before Map 2 is finalized, publish a compact **PS3 analysis-ready CSV** from the audited preparation workflow with the organization-level fields needed for PS3.

Recommended fields:

- Program_ID
- Canonical_Program_Name
- Current_League
- League_Acronym
- Affiliation_Season_Verified
- Affiliation_Evidence_Status
- Primary_County
- Primary_Municipality
- Home_Field
- Field_Address
- Latitude
- Longitude
- Geographic_QA_Category
- Geographic_Precision
- Analysis_Eligibility

Once that file is on GitHub, update `PS3_MASTER_URL`, rerun the notebook from the top, and verify that Maps 1-3 all use the same analysis-ready organization records.
